# Chiselling 101

`CC-BY 2026 Brooksbank, Kassabov, Wilson`

This notebook uses Dleto (Chisel) to extract null patterns in tensor data.  This notebooke is a demonstration of different chisels and their potential outputs.  

1. [Tucker Decompositions](#1-tucker-decompositions)
2. [Block Decompositions](#2-block-diagonalization)
3. [Stratification](#3-stratification)


## 1. Tucker Decompositions

We begin by using Dleto to reproduce the outcome of a familiar tensor structure known variously as a Tucker Decomposition or by names such as radical and total zero-divisor detection.  Tucker decompositions are essentially the tensor generalizations of locating the nullspace of a matrix and as such the idea is far older than its name and many specialized algorithms are available to perform this efficiently.  The use of chisels is not optimal but used here to familarize the user with chiselling on a somewhat well standard configuration.

> A **Tucker Decomposition** of a tensor $\Gamma$ framed by *axes* (known also as *modes*, *legs*, or *indices*) $\mathbb{K}^{d_1},\ldots, \mathbb{K}^{d_{\ell}}$, is a subset $A\subset\{1,\ldots,\ell\}$ and a decomposition
> $$\forall a\in A,\qquad \mathbb{K}^{d_a}=E_a\oplus R_a$$
> such that the contraction of $\Gamma$ on $R_a$ is 0.  

The decompositions can be given by partitioned bases of $E_a$ and $R_a$, or equivalently by linear projections $e_a:\mathbb{K}^{d_a}\to \mathbb{K}^{e_a}$ with kernel $R_a$.  In many situations the purpose of a Tucker decomposition is to restrict the tensor to the $E_a$-spaces, which we call the "truncated" form. 

**Performance Remark.** Tucker decompositions have been explored since the 1800's. There are many optimized strategies to discover them. This tutorial uses a familiar problem to explore the range of Dleto chisels, and illustrates the use of the parameters in Dleto chiseling.  Dleto chiseling operates with a complexity slightly greater than many more direct strategies for Tucker decompositions.  For high-performance computations, Dleto automatically switches to alternative optimized strategies by calling `nondeg`. 

### 1.1 Loading Dleto

Start by loading `Dleto.jl`.  If this is your first time you may need to install auxiliary packages and possibly set up Julia for notebooks.  That is a one-time setup for most users, see instructions here or consider using the fully online Binder demonstration.

In [ ]:
# Uncomment and run the first time, if Dleto is not installed
# using Pkg

# Option 1: To install from remote repository, use:
# Pkg.add(url="https://github.com/thetensor-space/OpenDleto")

# Option 2: If cloned locally at PATH 
# Pkg.activate( "../../" ) 
# Pkg.instantiate()

If you have already added Dleto to your Julia packages begin by loading the necessary packages, `ITensors` for general tensor controls, `Plots` for visualization tools, and `Dleto` the primary package of chisel techniques.

In [ ]:
using ITensors
using Plots
using Dleto
using Dleto: ⊕  # Explicitly import ⊕ from Dleto to resolve ambiguity

### 1.2 Creating a Tensor Experiment

We create two tensors, one randomized for control, and one identical in size but with 2 rows, columns, and slices set to approximately zero.  We then randomize the experiment tensor by applying a change in coordinates.  The goal is to using chiseling to detect and recover the hidden zero rows/columns/slices.

> **Note** Julia being mathematically oriented accepts $\LaTeX$ styled commands with tab-completion.  For example to insert the Unicode character for $\Gamma$ use `\Gamma` in the code area followed by `tab` (or cut-and-paste a character you see somewhere else).  Or replace with an simpler string of characters of your liking.  One suggestion, `Dleto.jl` calculations make substantial use of tensors, matrices, and lists of matrices.  A convention that clearly indicates those roles will be an investment worth your time.  
>
> We will be using:
> * capitol Greek letters `Γ` (`\Gamma`), `Δ` (`\Delta`), `Σ` (`\Sigma`), `Ξ` (`\Xi`), `Υ` (`\Upsilon`), etc. for tensors
> * capital English letters `X`, `Y`, `Z` etc. for matrices
> * lower case Geek letters for real numbers
> * lower case English letters for integers
> * Plural for lists of data, for instance `Γs` and `Xs`.

If this is your pass through this section we recommend using the existing parameters.  At the end of the section we encourage you to revisit this experiment and change the parameters to witness the changes.  Some suggested options are provied as comments.

In [ ]:
tol = 1e-6             # How sensitive the chiseling is

# Creates a control tensor
ds = (7,6,5)            # Increase to (25,25,25)
Γ = randn(Float64, ds)  

as=(4,4,3)
Δ = randn(Float64, as) ⊕ zeros(Float64, ds .- as)

# Add noise to experiment tensor
# db = 0.001; tol = 10*db            # Add some noise, increase tolerance
# Δ += db*randn(Float64, ds)

side_by_side(Γ, Δ; left_title="Control Γ", right_title="Experiment Δ")

For larger tensors it may help to do a visualization instead, and we can do this with the following.

In [ ]:
p1 = plot_tensor(Γ; title="Control Γ", color=:blue)    # tol bounds how much noise is shown
p2 = plot_tensor(Δ; title="Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

Under the initial parameters, the control tensor very likely has few to no all zero axes whereas the experiment evidently does.  If you adjust the parameters and background noise you may witness changes that match your parameter set.  

### 1.4 Tensor Randomization.
The role of chiseling is to recover such anamolies without knowing ahead of time.  So we now randomize the experiment so as to obscure this data.  Note that the randomization applies random bases change so the original coordinates.  Dleto offers a few automatic randomization proceedures for convenience, but you can also create you own list of matrices to apply as randomization, for example, replace `X1s=[ A[1], A[2], A[3]]` where each `A[i]` is an `ITensor` matrix with dimensions matching `ds` and define `Γ_rand = Γ * X1s`. 

Note that Delto uses `ITensors.jl` for most of its tensor managment and offers a few convenient methods to automatically convert standard Julia arrays to `ITensors.jl`.  To convert back to arrays you may use `Array(data, inds(data))` but `ITensors` format offers multiple convenient features for tensors and organizes the information to avoid common errors so the ideal situation is to adjust to the use of `ITensors`.

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ)
@assert isapprox(Γ * X1s, Γ_rand)

Δ_rand, Y1s = randomize_tensor(Δ)
@assert isapprox(Δ * Y1s, Δ_rand)


It should now be much less obvious that the control and experiment are any different, which we can confirm with a side-by-side comparison.

In [ ]:
side_by_side(Γ_rand, Δ_rand;
            left_title="Control Γ", right_title="Experiment Δ_rand")

We may also compare the two randomized tensors as plots.

In [ ]:
p1 = plot_tensor(Γ_rand, tol; title="Randomized Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand, tol; title="Randomized Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

### 1.5 Chiselling for Tucker Decompositions

As promised we now use Dleto to locate the evidence of a Tucker decomposition.  The key command is `stratify`.  While stratify can be used with multiple parameters it is has default selections that often apply quite well as a first approximation of a chiselling problem.  

In [ ]:
# Γ_strat, X2s = stratify(Γ_rand)
Γ_strat, X2s = stratify(Γ_rand; tol=tol)   # specify tol to match noise level
@assert isapprox(Γ_rand*X2s, Γ_strat)
# Δ_strat, Y2s = stratify(Δ_rand)
Δ_strat, Y2s = stratify(Δ_rand; tol=tol)
@assert isapprox(Δ_rand*Y2s, Δ_strat)

The return is again a tensor-transform pair with the similar guarantee as given in our randomization.  You may see some extra information printed describing the total number "derivations" detected during stratificaiton.  For most instances where a Tucker decomposition is present you will detect several derivations.  Random dense tensors typically admit only 2 derivations.  This is a first hint that Dleto has uncovered some properties in our randomized tensors.

**Round 2?** If you have modified the parameters to add noise chances are that naive stratifciation no longer distinguishes the experiment tensor.  This is where you will need to begin considering how much tolerance to pass along to the stratification by adding the optional parameter `tol=tol` (the tolerance we set above).  Replace the original commands by the commented commands to include this tolerance adjustment.  You will know if your tolerance is adequate if the number derivations for the experiment is more than 2.

---

Let is look at the results.

In [ ]:
p1 = plot_tensor(Γ_strat, tol; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat, tol; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

If the dimension and noise are within detectable proportions you should see the experiment tensor now clusters nearlly all its values in one region.  And while the precise location of that region may shift from experiment to experiment, it should bare proportions similar to the initiall nonzero region in the experiment tensor. 

This is a good place to return to the start of this [section](#12-creating-a-tensor-experiment) and begin altering the parameters to see how the results change.  
 * First you may increase the dimension.  **Caution:** the complexity of a tensor is proportional to the volume.  Changing (5,5,5) to (25,25,25) increase the input by 125 times not 20 times!  So begin with a caution increase until you appreciate the impact on performance and memory.
 * Second, increase the background noise, and set the tolerence to match, for example `tol=2*db` is a generally appropriate range.

### 1.6 Optimial Tucker Decompositions

While Dleto can demonstrate Tucker decompositions it is known that for this problem there are algorithms with faster running times.  Dleto includes one such function under the name `nondeg` which accepts two modes `:full` to regroup the nonzeros inside the original space, and `:trunc` to truncate the detected zero rows, columns, slices, and etc.  In most situations it is profitable to use these accelorated methods first because they are relatively low cost and if they succeed they will lower the dimensions of all subsequent computations.

In [ ]:
Γ_nondeg, X3s = Dleto.nondeg(Γ_rand, mode=:full);  # use mode=:trunc to truncate the 0's in the Tucker decomposition
@assert isapprox(Γ_rand * X3s, Γ_nondeg)
Δ_nondeg, Y3s = Dleto.nondeg(Δ_rand, mode=:full);
@assert isapprox(Δ_rand * Y3s, Δ_nondeg)

p1 = plot_tensor(Γ_nondeg, tol; title="Tucker Control Γ", color=:blue)
p2 = plot_tensor(Δ_nondeg, tol; title="Tucker Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

In [ ]:
Δ_nondeg_trunc, Y4s = Dleto.nondeg(Δ_rand, mode=:trunc);
@assert isapprox(Δ_rand * Y4s, Δ_nondeg_trunc)

p1 = plot_tensor(Δ_nondeg, tol; title="Tucker Experiment Δ full", color=:blue)
p2 = plot_tensor(Δ_nondeg_trunc, tol; title="Tucker Experiment Δ truncated", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

It is helpful to note that Tucker decompositions are a closure, once applied, a second application achieves no new clusters (though it may reorder and rescale the one already identifies).  

In [ ]:
Δ_nondeg_trunc, _ = nondeg(Δ_rand, mode=:trunc);
Δ_nondeg2_trunc, _ = nondeg(Δ_nondeg_trunc, mode=:trunc);
@assert size(Δ_nondeg_trunc) == size(Δ_nondeg2_trunc)

At this point if you have not already changed paremeters we encourage you to revisit the openning expirement to do so.  Otherwise you are ready to look at new expirements.

-----

# 2. Block Diagonalization

One way to think of Tucker decompositions is that it identifies a block on the diagonal of a tensor.  One natural extension of this concept is to have multiple blocks on the diagonal.  So now we create such tensors and hide them to see if Dleto chiselling can recover what we hide.

In [ ]:
as=[3,2,7]
bs=[4,5,2] 
cs=[5,3,3]
ds=as+bs+cs

Γ = randn(ds...);  # a control tensor

Δ = randn(as...) ⊕ randn(bs...) ⊕ randn(cs...)

# Add noise to experiment tensor
# db = 0.001; tol = 10*db           # Add some noise, increase tolerance
# Δ += db*randn(Float64, ds)

p1 = plot_tensor(Γ; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

Now let us once more hide the structure by randomizing our tensor's frame of reference.

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ);
Δ_rand, Y1s = randomize_tensor(Δ);  # In theory Δ_rand = Δ*Xs

# Plotted these two tensors are essentially indistinguishable from each other.
p1 = plot_tensor(Γ_rand; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

At this point we have some visual assurance that our experiment's data is hidden, and not just by permuting the order, we have a genuinely more dense tensor.  Now let us attempt to discover some structure through generic chiselling.

In [ ]:
Γ_strat, X2s = stratify(Γ_rand; tol=tol)   # specify tol to match noise level
@assert isapprox(Γ_rand*X2s, Γ_strat)
Δ_strat, Y2s = stratify(Δ_rand; tol=tol)
@assert isapprox(Δ_rand*Y2s, Δ_strat)

The plots here are showing dots whose volume is proportional to scalar size, which can omit tiny values.  Inspecting the actual data tells the store in more detail.  In particular we do not manage to up a small tolerance we see that we have recovered the degeneracy planted in the experimental tensor.

In [ ]:
p1 = plot_tensor(Γ_strat, tol; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat, tol; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

Once more we have recovered some structure.  The order of our blocks has changed (and it may be helpful to interact with 3D plot to understand the structure).


Now as before we can revisit the problem and add noise, add more blocks, and change dimensions.

## 3 Stratification

In linear algebra there are numerous uses for diagonals, block diagonals, and triangular forms of matrices.  Shifting to tensors beyond matrices (i.e. with 3+ axes) we can at last encounter something new and unrelated to matrices: curves and surfaces.

As always we will introduce a controlled experiment.

In [ ]:
ds = (30,30,30)
Γ = randn(Float64, ds...)  # a control tensor

db = 0.0
tol = 1e-4

# An experiment tensor with a hidden surface.
# - Choose f(x,y,z) to define the surface
f(x,y,z) = x^2 + y^2 + z^2
# - Populate Δ near the surface defined by f
Δ = zeros(Float64, ds...);
for i in 1:ds[1], j in 1:ds[2], k in 1:ds[3]
    if abs( f(i,j,k) - 30^2 ) < 1 + 2*randn()
        Δ[i,j,k] = randn()
    end
end


# to avoid long wait times raise the threshold for plotting
p1 = plot_tensor(Γ; title="Control Γ", color=:blue)
p2 = plot_tensor(Δ; title="Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

Note that Dleto also supplies a family of convenience functions to generate tensors of this sort along with more configurable noise and distance functions.

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ)
@assert isapprox(Γ * X1s, Γ_rand)
Δ_rand, Y1s = randomize_tensor(Δ)
@assert isapprox(Δ * Y1s, Δ_rand)

The randomized views are typical.

In [ ]:

p1 = plot_tensor(Γ_rand; title="Randomized Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand; title="Randomized Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

In [ ]:
@time Γ_strat, X2s = stratify(Γ_rand; tol=1e-2)   # specify tol to match noise level
@assert isapprox(Γ_rand*X2s, Γ_strat)


Now since we are stratifying larger surfaces it makes sense to ignore simpler structures such as implicitly included Tucker decompositions.  So we first reduce any available Tucker decompositions and stratify the rest.

In [ ]:
Δ_nondeg, Y2s = Dleto.nondeg(Δ_rand, mode=:trunc);
@assert isapprox(Δ_rand*Y2s, Δ_nondeg)
@time Δ_strat, Z1s = stratify(Δ_nondeg; tol=1e-1)
@assert isapprox(Δ_nondeg*Z1s, Δ_strat)

In [ ]:
p1 = plot_tensor(Γ_strat; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))